# Bonus: Supervised Fine-Tuning Gemma 4 on a Sudoku Task

### An optional extra for the AMD Radeon GRPO workshop, built with [Unsloth](https://github.com/unslothai/unsloth)

The main notebook taught Gemma 4 with **reinforcement learning**: no answers, only a score. This short notebook shows the other approach from Part 0, **supervised fine-tuning (SFT)**: the model is shown correct answers and learns to imitate them.

**The task.** Puzzle sites and datasets store a Sudoku as an 81-character string, read row by row (`.` or `0` for blanks). The GRPO notebook's `strategy(board, initial)` takes a 9x9 list of lists instead. The model is fine-tuned to do that conversion. The base model usually writes a parser function instead of the board, or miscounts the digits; after a few minutes of SFT it outputs the board directly.

This suits SFT because the correct answer is cheap to produce: a script wrote all of them. The dataset is in this repository, under `data/`: 1,000 training and 50 test examples, generated by `data/make_sudoku_sft.py`.

> Expected running time: about 10 minutes. **Shut down the GRPO notebook's kernel first** (Kernel, then Shut Down Kernel) so its GPU memory is free.

In [ ]:
%%capture
# Gemma 4 needs timm for its vision and audio towers; the portal image (unsloth_llama.cpp) gets it here.
!pip install --no-deps --upgrade timm

In [ ]:
# Same Gemma 4 E2B base as the GRPO notebook, in 16-bit. Short examples, so a 1024-token context is plenty.
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "/models/unsloth/gemma-4-E2B-it",
    max_seq_length = 1024,
    load_in_4bit = False,
)

In [ ]:
# Each example is a chat: the user gives a puzzle string, the assistant answers with the board.
from datasets import load_dataset

train_dataset = load_dataset("json", data_files = "data/sudoku_sft_train.jsonl", split = "train")
test_dataset = load_dataset("json", data_files = "data/sudoku_sft_test.jsonl", split = "train")
print(train_dataset)
for message in train_dataset[0]["messages"]:
    print(f"--- {message['role']} ---\n{message['content']}")

In [ ]:
# Score the model on 20 held-out puzzles: a puzzle counts only if every one of the 81 cells is right.
import ast, re

def parse_board(response):
    match = re.search(r"board\s*=\s*(\[.*\])", response, re.S)
    try:
        return ast.literal_eval(match.group(1)) if match else None
    except (SyntaxError, ValueError):
        return None

def evaluate(n = 20, show = 1):
    examples = test_dataset.select(range(n))
    texts = [tokenizer.apply_chat_template(ex["messages"][:1], tokenize = False, add_generation_prompt = True)
             for ex in examples]
    tokenizer.tokenizer.padding_side = "left"  # Batched generation pads on the left
    inputs = tokenizer(text = texts, add_special_tokens = False, padding = True, return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 320, do_sample = False, use_cache = True)
    correct = 0
    for i, ex in enumerate(examples):
        response = tokenizer.decode(outputs[i][inputs["input_ids"].shape[1]:], skip_special_tokens = True)
        correct += parse_board(response) == ex["board"]
        if i < show:
            print(f"--- prompt ---\n{ex['messages'][0]['content']}\n--- model ---\n{response}\n")
    print(f"Exact boards: {correct}/{n}")

evaluate()

In [ ]:
# LoRA on the language layers only; the vision and audio towers stay frozen.
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 32,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [ ]:
# Render each chat with Gemma's template (keep the leading <bos>: dropping it makes Gemma ramble),
# then mask the user turn so the loss is computed only on the assistant's answer.
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

train_dataset = train_dataset.map(lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize = False)})

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer.tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        max_steps = 120,  # About one pass over the 1,000 examples
        warmup_steps = 5,
        learning_rate = 2e-4,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        logging_steps = 10,
        save_strategy = "no",
        seed = 3407,
        report_to = "none",
        output_dir = "outputs/sft",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part = "<|turn>user\n", response_part = "<|turn>model\n")

In [ ]:
# Unlike GRPO, the loss is meaningful here: it should drop quickly toward 0.01.
trainer.train()

In [ ]:
# Same 20 puzzles as before training.
evaluate()

In [ ]:
# Only the LoRA adapters are saved; the export options at the end of the GRPO notebook apply here too.
model.save_pretrained("sft_saved_lora")
tokenizer.save_pretrained("sft_saved_lora")

## SFT versus GRPO

| | SFT (this notebook) | GRPO (main notebook) |
| --- | --- | --- |
| Training data | Prompts with correct answers | Prompts only |
| Learning signal | Match the answer, token by token | Reward from running the output |
| Good when | Correct answers are cheap to produce | Checking an answer is easier than writing one |

In practice the two are often combined: a short SFT run teaches the output format, then RL improves the content.

## Exercises

1. Train on fewer examples (`train_dataset.select(range(100))`) and see how accuracy changes.
2. Lower `max_steps` to 30, or raise it to 250.
3. Add your own prompt wordings to `PROMPTS` in `data/make_sudoku_sft.py`, regenerate the data, and retrain.
4. Check the model is still a generalist: ask it an unrelated question, as the GRPO notebook does with `ask()`.